In [ ]:
import io
from PIL import Image
from pillow_heif import register_heif_opener
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, DocumentStream

: 

In [ ]:
# 1. 注册 HEIF 支持
register_heif_opener()

In [ ]:
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, FormatOption
from docling.datamodel.base_models import InputFormat

# 1. 创建 Pipeline 配置
pipeline_options = PdfPipelineOptions()
# 指定 OCR 语言为日语和英语
pipeline_options.ocr_options.lang = ["ja", "en"]

# 2. 正确映射配置到 IMAGE 格式
# 注意：这里使用 FormatOption 来包裹 pipeline_options
converter = DocumentConverter(
    format_options={
        InputFormat.IMAGE: FormatOption(pipeline_options=pipeline_options),
        InputFormat.PDF: FormatOption(pipeline_options=pipeline_options)
    }
)

In [ ]:
converter = DocumentConverter()

In [ ]:
# --- 内存转换阶段 ---
image = Image.open("./input/IMG_5701.heic")

# 创建一个内存字节流对象
memory_buffer = io.BytesIO()

# 将图片以 JPEG 格式保存到内存中
image.convert("RGB").save(memory_buffer, format="JPEG")

# 关键点：将指针移回流的开头，否则 Docling 读取不到内容
memory_buffer.seek(0)

# --- Docling 处理阶段 ---
# 构建 DocumentStream，告知文件名（用于识别后缀）和字节流
doc_stream = DocumentStream(name="input.jpg", stream=memory_buffer)

# 执行转换
result = converter.convert(doc_stream)

print(result.document.export_to_markdown())

In [ ]:

# 2. 将 HEIC 转换为 JPEG 字节流 (无需写入磁盘)
image = Image.open("./input/IMG_5701.heic")
img_byte_arr = io.BytesIO()
image.convert("RGB").save(img_byte_arr, format='JPEG')
img_byte_arr.seek(0)

# 3. 交给 Docling 处理
converter = DocumentConverter()
# 使用 DocumentStream 处理内存中的数据
doc_stream = DocumentStream(name="converted.jpg", stream=img_byte_arr)
result = converter.convert(doc_stream)

return result.document.export_to_markdown()